# Setup
including loading packages, defining helper plotting functions, and generating subfigures

NOTE: necessary to run if you are running anything else in this notebook

In [44]:
%load_ext autoreload
%autoreload 2

import argparse
import sys
import os

import pandas as pd
import seaborn as sns
from matplotlib.patches import Rectangle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
from PIL import Image, ImageDraw, ImageFont

from src import secondlevel_group_glm
from src import secondlevel_group_encoding
from src import secondlevel_individual_encoding
from src import helpers
from src import plotting_helpers

load = True #set load=True to load analysis files (need to have downloaded them beforehand). False will run all analyses before plotting them (will take longer)
generate_subfigures = True # will take much longer than downloading the subfigures to use in making the main figures
base_dir = '../' #automatically uses this project directory, can be set to a different location

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [45]:
def get_default_args(base_dir):
	args_dict = { 
				'model': None,
				'task':	'sherlock',
				'mask':	'ISC',
				'perc_top_voxels':	5,
				'space': 'MNI152NLin2009cAsym',
				'smoothing_fwhm': 3.0,
				'chunklen': 120,
				'population': 'NT',
				'feature_of_interest': None,
				'ISC_type':'leave_one_out',
				'dir': base_dir,
				'out_dir': os.path.join(base_dir,'analysis'),
				'figure_dir': os.path.join(base_dir,'figures','subfigures')
			}
	return args_dict
def construct_args(args_dict):
	parser = argparse.ArgumentParser()
	parser.add_argument('--model','-model',type=str,default=args_dict['model'])
	parser.add_argument('--task','-task',type=str,default=args_dict['task'])
	parser.add_argument('--mask','-mask',type=str, default=args_dict['mask']) #the mask that contains all masks of interest (overarching mask )
	parser.add_argument('--perc-top-voxels','-perc-top-voxels',type=int,default=args_dict['perc_top_voxels'])
	parser.add_argument('--space','-space',type=str,default=args_dict['space'])
	parser.add_argument('--smoothing-fwhm','-smoothing-fwhm',type=float,default=args_dict['smoothing_fwhm'])
	parser.add_argument('--chunklen','-chunklen',type=int,default=args_dict['chunklen'])
	parser.add_argument('--population','-population',type=str,default=args_dict['population'])
	parser.add_argument('--feature-of-interest','-feature-of-interest',type=str,default=args_dict['feature_of_interest'])
	parser.add_argument('--ISC-type','-ISC-type',type=str,default=args_dict['ISC_type'])

	parser.add_argument('--dir', '-dir', type=str,
						default=args_dict['dir'])
	parser.add_argument('--out_dir', '-output', type=str,
						default=args_dict['out_dir'])
	parser.add_argument('--figure_dir', '-figures', type=str,
						default=args_dict['figure_dir'])
	def filter_jupyter_args(args):
		# Filter out any args starting with --f= (Jupyter passes --f=<file>)
		return [arg for arg in args if not arg.startswith('--f=')]
	filtered_args = filter_jupyter_args(sys.argv)
	args = parser.parse_args(filtered_args[1:])  # [1:] to skip the script name
	return args

def plot_secondlevel_group_glm(base_dir,task,cmap='gray_inferno'):
	args_dict = get_default_args(base_dir)
	args_dict['task']=task
	args_dict['mask']=None
	args = construct_args(args_dict)

	### GROUP ANALYSIS
	secondlevel_group = secondlevel_group_glm.SecondLevelGroup(args)
	secondlevel_group.do_stats = False

	secondlevel_group.compile_data()
	secondlevel_group.plot_zscores(vmin=0,FDR_correction=True, threshold=0.01,symmetric_cbar=False,cmap=cmap) #
def plot_secondlevel_group_encoding(base_dir,model,plotting,feature_names,cmap='gray_inferno',diff_cmap='',mask='ISC',load=False,vmax=None,png_dpi=None):
	# vmax can be a list to plot the ind_product_measure maps at several color scales from one stats run
	# png_dpi: also save the ind_product_measure maps as .png at this dpi
	args_dict = get_default_args(base_dir)
	args_dict['model']=model
	args_dict['mask'] = mask
	args_dict['mask_name'] = mask
	if(model=='correlation'):
		args_dict['smoothing_fwhm'] = 6.0

	args = construct_args(args_dict)

	### GROUP ANALYSIS
	secondlevel_group = secondlevel_group_encoding.SecondLevelGroup(args)
	secondlevel_group.do_stats = False
	secondlevel_group.explainable_variance_mask = False
	secondlevel_group.feature_names = feature_names
	secondlevel_group.scale_by=''#'total_variance'
	if(not load):
		measures_to_load = ['performance']+plotting
		if 'preference_map' in plotting:
			measures_to_load.append('ind_product_measure')
		secondlevel_group.compile_data(measures=measures_to_load) #always need to do performance to get proper brain shape and affine
	if('preference_map' in plotting):
		secondlevel_group.compute_preference_maps(restricted=False)
	secondlevel_group.save_results()
	if('performance' in plotting):
		title = ''
		if(model=='correlation'):
			secondlevel_group.permutation_brainiak(load=load)
			title='Intersubject Correlation'
		else:
			secondlevel_group.stats_wilcoxon()
			title = 'Joint Encoding Model Performance'
		if model=='cross_subject':
			title = 'Cross Subject Encoding Model Performance'
		secondlevel_group.plot_performance('stats',threshold=0.00001,vmin=0,vmax=vmax,cmap=cmap,title=title)
	if('ind_product_measure' in plotting):
		secondlevel_group.stats_wilcoxon()
		for vmax_ in (vmax if isinstance(vmax,list) else [vmax]):
			secondlevel_group.plot_ind_product_measure('stats',threshold=0.00001,vmin=0,vmax=vmax_,cmap=cmap,png_dpi=png_dpi)
	if('added_variance' in plotting):
		secondlevel_group.plot_added_variance('raw',threshold=0.00001,vmin=0,vmax=vmax,cmap=cmap)
	if('unique_variance' in plotting):
		secondlevel_group.plot_unique_variance('raw',threshold=0.00001,vmin=0,vmax=vmax,cmap=cmap)
	if('spatial_correlation' in plotting):
		secondlevel_group.plot_spatial_correlation('raw',threshold=None,vmin=None,vmax=vmax,cmap=cmap)
	if('ind_feature_performance' in plotting):
		secondlevel_group.plot_ind_feature_performance('raw',threshold=0.01,vmin=0,vmax=vmax,cmap=cmap)
	if('preference_map' in plotting):
		secondlevel_group.plot_preference_maps(label='both',fontsize=6)
	if('difference' in plotting):
		secondlevel_group.plot_difference('raw',threshold=0.01,cmap=diff_cmap,vmax=vmax)
def get_secondlevel_individual(base_dir, model, population='NT'):
	args_dict = get_default_args(base_dir)
	args_dict['model']=model
	args_dict['population']=population
	args = construct_args(args_dict)

	secondlevel_ind = secondlevel_individual_encoding.SecondLevelIndividual(args)
	secondlevel_ind.localizer_contrasts = {'SIpointlights':{'interact&no_interact','interact-no_interact'},'social interaction':{'interact&no_interact','interact-no_interact'},'language':{'intact-degraded'}}
	
	## specifying the names of each mask
	MT = ['MT']
	ISC = ['ISC']
	STS = ['pSTS','aSTS']#'frontal_language']
	language =['pTemp','aTemp','frontal']
	language_ROI_names = ['pTemp','aTemp','frontal']
	secondlevel_ind.localizer_masks = {'interact&no_interact':MT,'interact-no_interact':STS,'intact-degraded':language, 'degraded-intact':STS,
							'motion pointlights':MT,'SI pointlights':STS, 'language':language, 'DMN':STS,
							'social interaction':STS,'social':STS,
       						'GPT2':STS,'sbert':language,'sbert+word2vec':language,'word2vec':language,'hubert':language,
							'motion':MT,'num_agents':STS, 'alexnet':STS,'alexnet':STS,
							'valence':STS,'face':STS,'mentalization':STS, 'arousal':STS,
							'speaking':STS,'indoor_outdoor':STS,
							'turn_taking':STS,'written_text':STS,'music':STS,'none':STS}
	secondlevel_ind.MT = MT
	secondlevel_ind.ISC = ISC
	secondlevel_ind.STS = STS
	secondlevel_ind.language = language
	secondlevel_ind.language_ROI_names = language_ROI_names
	return secondlevel_ind

In [46]:
#load the default arguments for easy access
args_dict = get_default_args(base_dir)
subject = 'sub-33' #specifying representative subject #sub-26, sub-39 are other options

In [47]:
def label_brain_plots(output_filename, final_image, labels_dict={}, desc_dict={}, feature_names=[], features_on_width=True,pad_prop=0,include_desc=True,include_ind_group_labels=True,color_dict=None):
    # add padding for the labels
    pad = 200
    if features_on_width:
        padding = (pad, pad, pad, pad)
    else:
        padding = (pad, pad, pad, 0)
    
    new_width = final_image.width + padding[0] + padding[2]
    new_height = final_image.height + padding[1] + padding[3]

    new_image = Image.new('RGBA', (new_width, new_height), (255, 255, 255, 0))
    new_image.paste(final_image, (padding[0], padding[1]))

    draw = ImageDraw.Draw(new_image)
    font = ImageFont.truetype("Arial.ttf", 85)
    label_font = ImageFont.truetype("Arial.ttf", 125)  # Larger font for 'Individual' and 'Group'
    width, height = final_image.size
    
    if features_on_width:
        portion = width // len(feature_names)
        centers = [(portion * i + portion // 2.35, 15) for i in range(len(feature_names))]
    else:
        portion = height // len(feature_names)
        centers = [(new_width // 2, (portion * i)-pad_prop*portion/3) for i in range(len(feature_names))]
    
    text_padding = 25
    if(len(feature_names)>1):
        for center, feature in zip(centers, feature_names):
            bbox = draw.textbbox((0, 0), labels_dict[feature], font=font)
            text_width = bbox[2] - bbox[0]
            text_height = bbox[3] - bbox[1]
            
            if features_on_width:
                text_x = center[0] - (text_width // 2) + pad
                text_y = center[1] + (text_height // 2)
                rectangle_x = center[0] - portion // len(feature_names)
                rectangle_coords = (
                    rectangle_x + pad,                # Left
                    text_y - text_padding,                     # Top
                    rectangle_x + (portion//len(feature_names)*2) +pad ,        # Right
                    text_y + text_padding*5      # Bottom
                    )
            else:
                text_x = center[0] - (text_width // 2)
                text_y = center[1] - (text_height // 2) + pad
                rectangle_x = center[0] - (text_width //2)
                rectangle_coords = (
                    rectangle_x - pad,                # Left
                    text_y - text_padding,                     # Top
                    rectangle_x + text_width+pad ,        # Right
                    text_y + text_padding*5      # Bottom
                    )
            
            draw.rounded_rectangle(rectangle_coords, radius=30, width=10,fill='lightgray')
            draw.text((text_x,text_y), labels_dict[feature], fill="black", font=font)
            
            if(include_desc):
                bbox = draw.textbbox((0, 0), desc_dict[feature], font=font)
                text_width = bbox[2] - bbox[0]
                text_height = bbox[3] - bbox[1]
                
                if features_on_width:
                    text_x = center[0] - (text_width // 2) + pad
                    text_y = new_image.size[1] - text_height * 1.6
                else:
                    text_x = center[0] - text_width // 2
                    text_y = center[1]  + portion
                
                draw.text((text_x, text_y), desc_dict[feature], fill="black", font=font)
            
    
    if(include_ind_group_labels):
        if features_on_width:
            #rotated labels for labeling Individual and Group brain maps
            for y,text in zip((1,3),['Individual','Group']):
                bbox = draw.textbbox((0, 0), text, font=font)
                text_width = bbox[2] - bbox[0]
                text_height = bbox[3] - bbox[1]
                text_image = Image.new('RGBA', (text_width,text_height+20), (255, 255, 255, 0)) #transparent background
                draw = ImageDraw.Draw(text_image)
                text_position = (0,0)
                draw.text(text_position, text, font=font, fill="black")
                text_image = text_image.rotate(90, resample=Image.BICUBIC, expand=1)
                new_image.paste(text_image, (-15, (final_image.size[1]//4)*y), text_image)
        else:
        # Labels for 'Individual' and 'Group' along the top
            label_positions = [(0.24, 'Individual'), (0.7, 'Group')]
            for pos, text in label_positions:
                bbox = draw.textbbox((0, 0), text, font=label_font)
                text_width = bbox[2] - bbox[0]
                text_x = int(new_image.size[0] * pos - text_width / 2)
                text_y = new_height - (pad)
                draw.text((text_x, text_y), text, font=label_font, fill="black")

    new_image.save(output_filename)
    return new_image

# Running second level analysis and generating subfigures 
NOTE: if you cloned the repository this is not necessary since the subfigures already exist, but it is possible to re-run the second level analyses here. Set `load` and `generate_subfigures` in Setup.

**From a clone of this repository (`load=True`):** the cached second level results that these cells load are in the repository, so they run without downloading anything:
- Main Figure 3B (joint model variance decomposition in the localized regions) and Supplementary Tables S2 and S3
- Main Figure 4 (encoding weight similarity across regions) and Supplementary Tables S4–S7
- Main Figure 5 (pooled weight PCA interpretation)
- Supplementary Figures 5, 6, 7 and 8 (supplementary ROI analyses)
- Supplementary Figure 9 (full movie vs. speaking-only evaluation)
- the overlap between the social interaction and language regions (reported in the text)

The statistics are recomputed from the cached values with pymer4, which needs R (installed with `environment.yml`).

**Needs data from Hugging Face** (https://huggingface.co/datasets/hsmall/sherlock-and-localizers-fMRI), because these cells read every participant's maps:
- Main Figure 2 and the group column of Supplementary Figure 3 (group maps; this cell always runs with `load=False`): `analysis/EncodingModel/performance`, `analysis/EncodingModel/ind_product_measure`
- Main Figure 3A and Supplementary Figure 4 (preference maps): `analysis/EncodingModel/ind_product_measure`, `analysis/SecondLevelIndividual/localizer_masks`
- Supplementary Figure 1 (localizer maps): `analysis/GLM`
- Supplementary Figure 3, example participant column: `analysis/EncodingModel/ind_product_measure`
- Supplementary Figure 10 (feature space dimensionality): `features`

**Running everything from scratch (`load=False`):** needs the full `analysis` directory (`EncodingModel`, `GLM`, `Brain2Brain`, `SecondLevelIndividual`, `SecondLevelGroup`) and `features` from Hugging Face. Running all of the analyses will take some time.

### Group maps of cross-subject encoding and joint feature encoding model performance and feature performance within joint model

In [ ]:
load = False #load is set in Setup! you can change it here though
vmax = 0.18
if generate_subfigures:
    # plot_secondlevel_group_encoding(base_dir,'joint_chunk120',plotting=['performance'],feature_names=[],cmap='inferno_symmetric',vmax=vmax)
    # plot_secondlevel_group_encoding(base_dir,'cross_subject',plotting=['performance'],feature_names=[],load=load,cmap='inferno_symmetric',vmax=None)

    # individual feature predictivity group maps (Main Figure 2C-D at vmax 0.1, Supplementary Figure 3 at vmax None, saved as png too)
    # and the group preference map (Main Figure 3A). The feature order sets the volume order of the saved
    # ind_product_measure and product_measure_p_fdr files (and the preference map values 1-4), so keep one order for all of them
    feature_names = ['alexnet_eps0.3','hubert_chunk120','motion','GPT2_chunk120']
    plot_secondlevel_group_encoding(base_dir,'joint_chunk120',plotting=['ind_product_measure','preference_map'],feature_names=feature_names,load=load,cmap='inferno_symmetric',vmax=[0.1,None],png_dpi=600) #saves in subfigures folder


self.ind_product_measure (4, 88767)
temp (5, 97, 115, 97)
temp_ind_product_measure (5, 163842)
0
pref1_map_surf (163842,)
self.ind_product_measure (4, 88767)
temp (5, 97, 115, 97)
temp_ind_product_measure (5, 163842)
0
pref1_map_surf (163842,)
0
ind_product_measure
(4, 88767)
saved: ind_product_measure
(4, 97, 115, 97)
(88767,)
(97, 115, 97)
saved: preference1_map
(97, 115, 97)
(88767,)
saved: preference2_map
(97, 115, 97)


### Overlap between social interaction and language ROIs (reported in text, no figure)

In [12]:
def get_overlap(base_dir,masks,perc_top_voxels,load=False,file_tag=''):
    """This function sets up all required analyses for the overlap between social interaction and language regions to be computed. 
        It runs the glm voxel selection for social interaction and language both within the STS parcel, for the most conservative 
        estimate of overlap. 
        If there are multiple masks, the voxel selection happens within in each and then the overlap is computed on their union. """
    secondlevel = get_secondlevel_individual(base_dir,'joint_chunk120')
    secondlevel.perc_top_voxels=str(perc_top_voxels)
    
    enc_localizer_label_dict = {}
    glm_localizer_label_dict = {}
    file_tag = file_tag
    regions = []
    secondlevel = get_secondlevel_individual(base_dir, 'joint_chunk120')
    secondlevel.perc_top_voxels=str(perc_top_voxels)
    #runs glm and encoding voxel selection again with the same mask across both domains
    glm_localizers_to_plot = ['social interaction', 'language']

    all_localizers = glm_localizers_to_plot
    for localizer in all_localizers:
        secondlevel.localizer_masks[localizer]=masks
        glm_localizer_label_dict[localizer] = 'STS' #making all localizers in the STS
    secondlevel.localizer_masks['interact-no_interact']=masks
    secondlevel.localizer_masks['intact-degraded']=masks

    secondlevel.localizer_contrasts = {'SIpointlights':{'interact-no_interact'},'social interaction':{'interact-no_interact'},'language':{'intact-degraded'}}
    secondlevel.language = masks #putting all localizers (social and language) in STS for most conservative overlap measuremment
    secondlevel.language_ROI_names = masks
    secondlevel.glm_voxel_selection(load=load,plot=False,plot_ind=False,response_label='performance',plot_stacked=False,localizers_to_plot=glm_localizers_to_plot,localizer_label_dict=glm_localizer_label_dict,filepath_tag=file_tag)
    if(not load):
        secondlevel.generate_binary_localizer_maps_glm(glm_task='SIpointlights',plot=False)
        secondlevel.generate_binary_localizer_maps_glm(glm_task='language',plot=False)
    regions.extend([('interact-no_interact','glm-SIpointlights','NA','_'.join(masks)),('intact-degraded','glm-language','NA','_'.join(masks))])
    secondlevel.compute_all_overlap2(load=load,regions=regions,label='_'.join(masks),file_tag=file_tag)

In [15]:
load = load
masks = ['pSTS','aSTS']
perc_top_voxels = args_dict['perc_top_voxels']
if generate_subfigures:
	for mask in masks: #compute the overlap separately in each mask. 
		get_overlap(base_dir=base_dir,load=load,masks=[mask], perc_top_voxels=perc_top_voxels,file_tag='_for_overlap')

Running GLM voxel selection...


('language', '12', 'degraded', 'sub-05')
('language', '12', 'degraded', 'sub-05')
('language', '12', 'intact', 'sub-05')
('language', '12', 'degraded', 'sub-06')


('language', '12', 'degraded', 'sub-06')


('language', '12', 'intact', 'sub-06')
('language', '12', 'degraded', 'sub-07')


('language', '12', 'intact', 'sub-07')
('language', '12', 'intact', 'sub-07')
('language', '12', 'intact', 'sub-05')
('language', '12', 'degraded', 'sub-08')
('language', '12', 'degraded', 'sub-08')
('language', '12', 'intact', 'sub-08')
('language', '12', 'intact', 'sub-06')


('language', '12', 'intact', 'sub-08')
('language', '12', 'degraded', 'sub-09')
('language', '12', 'degraded', 'sub-09')
('language', '12', 'degraded', 'sub-07')
('language', '12', 'intact', 'sub-09')
('language', '12', 'intact', 'sub-09')
('language', '12', 'degraded', 'sub-10')
('language', '12', 'degraded', 'sub-10')
('language', '12', 'intact', 'sub-10')


... (14 outputs left out)

Saved CSV to ../analysis/SecondLevelIndividual/sub-NT_smoothingfwhm-3.0_chunklen-20_mask-ISC_model-joint_perc_top_voxels-5_glm_localizer_performance__for_overlap_summary.csv
Saved PKL to ../analysis/SecondLevelIndividual/sub-NT_smoothingfwhm-3.0_chunklen-20_mask-ISC_model-joint_perc_top_voxels-5_glm_localizer_performance__for_overlap_voxelwise.pkl
generating binary glm voxel selection maps:


generating binary glm voxel selection maps:


computing overlap between voxel groups:


[LEFT] ('interact-no_interact', 'glm-SIpointlights', 'NA', 'aSTS') vs ('intact-degraded', 'glm-language', 'NA', 'aSTS'): mean overlap = 0.032, std = 0.039, n = 25
[RIGHT] ('interact-no_interact', 'glm-SIpointlights', 'NA', 'aSTS') vs ('intact-degraded', 'glm-language', 'NA', 'aSTS'): mean overlap = 0.038, std = 0.076, n = 25


### Joint model variance decomposition within motion, social interaction, and language ROIs

In [25]:
load = True ## load precomputed results while plotting
secondlevel_ind = get_secondlevel_individual(base_dir,'joint_chunk120',population='NT')
features = secondlevel_ind.feature_names
glm_localizers_to_plot = ['motion','social interaction', 'language'] 
glm_localizer_label_dict = {'motion':'MT','social interaction':'STS','language':'language','DMN':'STS'}

#average over all layers of any relevant DNNs (collapse all layers down to one feature name)
features_no_layer = [feature if 'layer' not in feature else feature.split('_layer')[0] for feature in features]
features = list(dict.fromkeys(features_no_layer)) #get rid of duplicates
features_final = [feature for feature in features ]#if 'SimCLR' not in feature]
features = features_final
secondlevel_ind.feature_names = features
secondlevel_ind.plot_features = features
secondlevel_ind.scale_by = '' #not scaling the results by anything (could scale by total variance explained with 'total_variance')
response_label = 'ind_product_measure' #extracting the product measure of each feature
average_posterior_anterior = True #average pSTS and aSTS (social interaction region) together, pTemp and aTemp (language regions) together

print(features)

perc_top_voxels = args_dict['perc_top_voxels']
if generate_subfigures:
    filepath_tag = ''
    figure_tag='all'

    ### the main variance decomposition results ###
    ## glm_voxel_selection plots multiple subfigures: 
    ## - the stacked bar plot of the variance explained by each feature in the joint model for each localized region (from encoding model)
    ## - the responses to each condition of the localizer experiment in each localized region (from glm)
#   
    # figure_tag='sep_post_ant'
    plotting_helpers.figure_width_mm = 108
    plotting_helpers.figure_fontsize = 5
    csv_path,pkl_path = secondlevel_ind.glm_voxel_selection(load=True,include_cross_subject=False,legend_below=True,label_region=True,restrict_legend=True,average_posterior_anterior=average_posterior_anterior,average_left_right=False,stats_to_do='compare_to_zero',plot_stacked=True,plot_ind=True,response_label=response_label,localizers_to_plot=glm_localizers_to_plot,localizer_label_dict=glm_localizer_label_dict,filepath_tag=filepath_tag,figure_tag=figure_tag,save_source_data=True)

    # ### results separately plotting the product measure of each layer in alexnet ###
    # secondlevel_ind = get_secondlevel_individual(base_dir,'vislang_gpt2_chunk120_eps0.3',population='NT')
    # filepath_tag='layers'
    # # # average_posterior_anterior=True
    # secondlevel_ind.plot_features = ['alexnet_eps0.3_layer'+str(layer) for layer in [1,2,3,4,5,6,7]] + ['GPT2_chunk120_layer'+str(layer) for layer in [1,2,3,4,5,6,7,8,9,10,11,12]] + ['hubert_chunk120_layer'+str(layer) for layer in [1,2,3,4,5,6,7,8,9,10,11,12]]
    # # # # # # secondlevel_ind.plot_features = ['hubert_chunk60_layer'+str(layer) for layer in [1,2,3,4,5,6,7,8,9,10,11,12]]
    # # # # # # secondlevel_ind.plot_features = ['SimCLR_attention_layer'+str(layer) for layer in [1,2,3,4,5,6,7,8,9,10,11,12]] +['SimCLR_embedding_layer'+str(layer) for layer in [1,2,3,4,5,6,7,8,9,10,11,12]]
    # secondlevel_ind.plot_features = ['GPT2_chunk120_layer'+str(layer) for layer in [1,2,3,4,5,6,7,8,9,10,11,12]]

    # figure_tag='GPT2'
    # secondlevel_ind.glm_voxel_selection(average_left_right=False,load=True,include_isc_and_cross_subject=False,restrict_legend=True,average_posterior_anterior=average_posterior_anterior,stats_to_do=None,plot_stacked=True,plot_ind=True,response_label=response_label,localizers_to_plot=glm_localizers_to_plot,localizer_label_dict=glm_localizer_label_dict,filepath_tag=filepath_tag,figure_tag=figure_tag)
    # secondlevel_ind.plot_features = ['hubert_chunk120_layer'+str(layer) for layer in [1,2,3,4,5,6,7,8,9,10,11,12]]
    # figure_tag='hubert'
    # secondlevel_ind.glm_voxel_selection(average_left_right=False,load=True,include_isc_and_cross_subject=False,restrict_legend=True,average_posterior_anterior=average_posterior_anterior,stats_to_do=None,plot_stacked=True,plot_ind=True,response_label=response_label,localizers_to_plot=glm_localizers_to_plot,localizer_label_dict=glm_localizer_label_dict,filepath_tag=filepath_tag,figure_tag=figure_tag)
    # secondlevel_ind.plot_features = ['alexnet_eps0.3_layer'+str(layer) for layer in [1,2,3,4,5,6,7]]
    # figure_tag='alexnet'
    # secondlevel_ind.glm_voxel_selection(average_left_right=False,load=True,include_isc_and_cross_subject=False,restrict_legend=True,average_posterior_anterior=average_posterior_anterior,stats_to_do=None,plot_stacked=True,plot_ind=True,response_label=response_label,localizers_to_plot=glm_localizers_to_plot,localizer_label_dict=glm_localizer_label_dict,filepath_tag=filepath_tag,figure_tag=figure_tag)
    
    # figure_tag='hemis_averaged'
    # secondlevel_ind.glm_voxel_selection(load=load,restrict_legend=True,average_left_right=True,average_posterior_anterior=average_posterior_anterior,stats_to_do=None,plot_stacked=False,plot_ind=True,response_label=response_label,localizers_to_plot=glm_localizers_to_plot,localizer_label_dict=glm_localizer_label_dict,filepath_tag=filepath_tag,figure_tag=figure_tag)

['alexnet_eps0.3', 'motion', 'hubert_chunk120', 'word2vec_chunk120', 'GPT2_chunk120', 'pixel', 'hue', 'face', 'num_agents', 'social', 'valence', 'arousal', 'pitch', 'amplitude', 'music', 'speaking', 'turn_taking', 'mentalization', 'written_text']
Running GLM voxel selection...
Linear mixed model fit by REML [’lmerMod’]
Formula: glm_weight~1+Condition+(1|subject)

Family: gaussian	 Inference: parametric

Number of observations: 68	 Groups: {'subject': 34.0}

Log-likelihood: -40.041 	 AIC: 88.083

Random effects:

                 Name    Var    Std
subject   (Intercept)  0.488  0.698
Residual               0.031  0.176

No random effect correlations specified

Fixed effects:

Linear mixed model fit by REML [’lmerMod’]
Formula: glm_weight~1+Condition+(1|subject)

Family: gaussian	 Inference: parametric

Number of observations: 50	 Groups: {'subject': 25.0}

Log-likelihood: 9.541 	 AIC: -11.082

Random effects:

                 Name    Var    Std
subject   (Intercept)  0.032  0.179
Resid

Linear mixed model fit by REML [’lmerMod’]
Formula: glm_weight~1+Condition+(1|subject)

Family: gaussian	 Inference: parametric

Number of observations: 68	 Groups: {'subject': 34.0}

Log-likelihood: 10.759 	 AIC: -13.519

Random effects:

                 Name    Var    Std
subject   (Intercept)  0.042  0.206
Residual               0.015  0.120

No random effect correlations specified

Fixed effects:

Linear mixed model fit by REML [’lmerMod’]
Formula: glm_weight~1+Condition+(1|subject)

Family: gaussian	 Inference: parametric

Number of observations: 50	 Groups: {'subject': 25.0}

Log-likelihood: -4.706 	 AIC: 17.412

Random effects:

                 Name    Var    Std
subject   (Intercept)  0.036  0.190
Residual               0.036  0.189

No random effect correlations specified

Fixed effects:

Linear mixed model fit by REML [’lmerMod’]
Formula: glm_weight~1+Condition+(1|subject)

Family: gaussian	 Inference: parametric

Number of observations: 68	 Groups: {'subject': 34.0}

Log-l

Linear mixed model fit by REML [’lmerMod’]
Formula: glm_weight~1+Condition+(1|subject)

Family: gaussian	 Inference: parametric

Number of observations: 50	 Groups: {'subject': 25.0}

Log-likelihood: 32.768 	 AIC: -57.536

Random effects:

                 Name    Var    Std
subject   (Intercept)  0.021  0.143
Residual               0.004  0.062

No random effect correlations specified

Fixed effects:

Linear mixed model fit by REML [’lmerMod’]
Formula: glm_weight~1+Condition+(1|subject)

Family: gaussian	 Inference: parametric

Number of observations: 50	 Groups: {'subject': 25.0}

Log-likelihood: -32.382 	 AIC: 72.764

Random effects:

                 Name    Var    Std
subject   (Intercept)  0.094  0.306
Residual               0.125  0.353

No random effect correlations specified

Fixed effects:

**NOTE**: Column for 'residuals' not created in model.data, but saved in model.resid only. This is because you have rows with NaNs in your data.

**NOTE** Column for 'fits' not created in 

Linear mixed model fit by REML [’lmerMod’]
Formula: glm_weight~1+Condition+(1|subject)

Family: gaussian	 Inference: parametric

Number of observations: 68	 Groups: {'subject': 34.0}

Log-likelihood: -40.041 	 AIC: 88.083

Random effects:

                 Name    Var    Std
subject   (Intercept)  0.488  0.698
Residual               0.031  0.176

No random effect correlations specified

Fixed effects:

Linear mixed model fit by REML [’lmerMod’]
Formula: glm_weight~1+Condition+(1|subject)

Family: gaussian	 Inference: parametric

Number of observations: 50	 Groups: {'subject': 25.0}

Log-likelihood: 9.541 	 AIC: -11.082

Random effects:

                 Name    Var    Std
subject   (Intercept)  0.032  0.179
Residual               0.015  0.122

No random effect correlations specified

Fixed effects:

Linear mixed model fit by REML [’lmerMod’]
Formula: glm_weight~1+Condition+(1|subject)

Family: gaussian	 Inference: parametric

Number of observations: 68	 Groups: {'subject': 34.0}

Log-l

boundary (singular) fit: see help('isSingular') 

Linear mixed model fit by REML [’lmerMod’]
Formula: encoding_response~0+enc_feature_name+(1|subject)

Family: gaussian	 Inference: parametric

Number of observations: 475	 Groups: {'subject': 25.0}

Log-likelihood: 1322.889 	 AIC: -2603.779

Random effects:

                 Name  Var    Std
subject   (Intercept)  0.0  0.000
Residual               0.0  0.012

No random effect correlations specified

Fixed effects:

Linear mixed model fit by REML [’lmerMod’]
Formula: encoding_response~0+enc_feature_name+(1|subject)

Family: gaussian	 Inference: parametric

Number of observations: 475	 Groups: {'subject': 25.0}

Log-likelihood: 1286.569 	 AIC: -2531.138

Random effects:

                 Name  Var    Std
subject   (Intercept)  0.0  0.001
Residual               0.0  0.013

No random effect correlations specified

Fixed effects:



boundary (singular) fit: see help('isSingular') 

Linear mixed model fit by REML [’lmerMod’]
Formula: encoding_response~0+enc_feature_name+(1|subject)

Family: gaussian	 Inference: parametric

Number of observations: 475	 Groups: {'subject': 25.0}

Log-likelihood: 1262.306 	 AIC: -2482.611

Random effects:

                 Name  Var    Std
subject   (Intercept)  0.0  0.000
Residual               0.0  0.014

No random effect correlations specified

Fixed effects:

**NOTE**: Column for 'residuals' not created in model.data, but saved in model.resid only. This is because you have rows with NaNs in your data.

**NOTE** Column for 'fits' not created in model.data, but saved in model.fits only. This is because you have rows with NaNs in your data.

Linear mixed model fit by REML [’lmerMod’]
Formula: encoding_response~0+enc_feature_name+(1|subject)

Family: gaussian	 Inference: parametric

Number of observations: 475	 Groups: {'subject': 24.0}

Log-likelihood: 1110.687 	 AIC: -2179.374

Rand

Linear mixed model fit by REML [’lmerMod’]
Formula: encoding_response~0+enc_feature_name+(1|subject)

Family: gaussian	 Inference: parametric

Number of observations: 475	 Groups: {'subject': 25.0}

Log-likelihood: 1344.313 	 AIC: -2646.627

Random effects:

                 Name  Var    Std
subject   (Intercept)  0.0  0.002
Residual               0.0  0.012

No random effect correlations specified

Fixed effects:

Linear mixed model fit by REML [’lmerMod’]
Formula: encoding_response~0+enc_feature_name+(1|subject)

Family: gaussian	 Inference: parametric

Number of observations: 475	 Groups: {'subject': 25.0}

Log-likelihood: 1304.795 	 AIC: -2567.589

Random effects:

                 Name  Var    Std
subject   (Intercept)  0.0  0.002
Residual               0.0  0.013

No random effect correlations specified

Fixed effects:

boundary (singular) fit: see help('isSingular') 

Linear mixed model fit by REML [’lmerMod’]
Formula: encoding_response~0+enc_feature_name+(1|subject)

Family: ga

Running ANOVA for left hemisphere per region pair
  Testing pair: left_MT vs left_STS
Linear mixed model fit by REML [’lmerMod’]
Formula: encoding_response~1+Feature_Space+hemi_mask+(1|subject)

Family: gaussian	 Inference: parametric

Number of observations: 950	 Groups: {'subject': 25.0}

Log-likelihood: 2743.510 	 AIC: -5443.020

Random effects:

                 Name  Var    Std
subject   (Intercept)  0.0  0.002
Residual               0.0  0.012

No random effect correlations specified

Fixed effects:

Linear mixed model fit by REML [’lmerMod’]
Formula: encoding_response~1+Feature_Space*hemi_mask+(1|subject)

Family: gaussian	 Inference: parametric

Number of observations: 950	 Groups: {'subject': 25.0}

Log-likelihood: 2669.663 	 AIC: -5259.326

Random effects:

                 Name  Var    Std
subject   (Intercept)  0.0  0.002
Residual               0.0  0.012

No random effect correlations specified

Fixed effects:

  Testing pair: left_MT vs left_temporal
Linear mixed model fi

Running ANOVA for right hemisphere per region pair
  Testing pair: right_MT vs right_STS
Linear mixed model fit by REML [’lmerMod’]
Formula: encoding_response~1+Feature_Space+hemi_mask+(1|subject)

Family: gaussian	 Inference: parametric

Number of observations: 950	 Groups: {'subject': 25.0}

Log-likelihood: 2662.189 	 AIC: -5280.378

Random effects:

                 Name  Var    Std
subject   (Intercept)  0.0  0.002
Residual               0.0  0.013

No random effect correlations specified

Fixed effects:

Linear mixed model fit by REML [’lmerMod’]
Formula: encoding_response~1+Feature_Space*hemi_mask+(1|subject)

Family: gaussian	 Inference: parametric

Number of observations: 950	 Groups: {'subject': 25.0}

Log-likelihood: 2594.907 	 AIC: -5109.814

Random effects:

                 Name  Var    Std
subject   (Intercept)  0.0  0.002
Residual               0.0  0.013

No random effect correlations specified

Fixed effects:

  Testing pair: right_MT vs right_temporal
**NOTE**: Column

### Preference brain maps

In [29]:
## NOTE: The ROI files that are overlaid on the preference maps are generated during the joint model variance decomposition analyses, so that needs to have run first!

if generate_subfigures:
    secondlevel_ind = get_secondlevel_individual(base_dir,'joint_chunk120')
    # secondlevel_ind.subjects['sherlock'] = ['sub-32','sub-61']

    # generate combined mask glm binary masks
    secondlevel_ind.generate_binary_localizer_maps_glm(plot=False,glm_task='SIpointlights')
    secondlevel_ind.generate_binary_localizer_maps_glm(plot=False,glm_task='language')

    # preference map plotting
    plotting = ['preference_map']
    features = secondlevel_ind.feature_names
    features_no_layer = [feature if 'layer' not in feature else feature.split('_layer')[0] for feature in features] 
    features = features_no_layer

    # NOTE: there will be some errors here because not all subjects completed the language localizer (which is overlaid on the preference map)
    # this is fine, a preference map will still be generated but with only the social interaction regions overlaid

    #generate preference maps of only the ventral and lateral view for the main figure
    file_tag = 'vent_lat' 
    secondlevel_ind.generate_preference_maps(width_mm=72, fontsize=6, measure='ind_product_measure',views=['lateral','ventral'],restricted=True,threshold=0.0001,features=features,file_tag=file_tag)



generating binary glm voxel selection maps:


generating binary glm voxel selection maps:


No such file or no access: '../analysis/SecondLevelIndividual/localizer_masks/sub-05_smoothingfwhm-3.0_mask-pTemp_aTemp_frontal_glm_loc-intact-degraded_run-all_binary.nii.gz'
No such file or no access: '../analysis/SecondLevelIndividual/localizer_masks/sub-06_smoothingfwhm-3.0_mask-pTemp_aTemp_frontal_glm_loc-intact-degraded_run-all_binary.nii.gz'
No such file or no access: '../analysis/SecondLevelIndividual/localizer_masks/sub-07_smoothingfwhm-3.0_mask-pTemp_aTemp_frontal_glm_loc-intact-degraded_run-all_binary.nii.gz'
No such file or no access: '../analysis/SecondLevelIndividual/localizer_masks/sub-08_smoothingfwhm-3.0_mask-pTemp_aTemp_frontal_glm_loc-intact-degraded_run-all_binary.nii.gz'
No such file or no access: '../analysis/SecondLevelIndividual/localizer_masks/sub-09_smoothingfwhm-3.0_mask-pTemp_aTemp_frontal_glm_loc-intact-degraded_run-all_binary.nii.gz'
No such file or no access: '../analysis/SecondLevelIndividual/localizer_masks/sub-10_smoothingfwhm-3.0_mask-pTemp_aTemp_front

### Analyses of encoding model weights: feature tuning similarity across regions
NOTE: to run these from scratch, please download all of the individual participant weight files from hugging face. 

In [39]:
### Main Figure 4: response similarity (Spearman correlation between the fitted encoding model weights) between the localized regions
## one matrix per feature space; posterior and anterior regions are averaged, and hemispheres are averaged
load = True
secondlevel_ind = get_secondlevel_individual(base_dir,'vislang_gpt2_chunk120_eps0.3')
## feature space: (features whose weights are correlated, filepath_tag of the cached results)
## the last three layers of each DNN, as computed on the cluster with python -m src.response_similarity (one call per feature space)
## with load=True only the feature space name is used, to read the cached results
response_similarity_features = {'motion': (['motion'],'motion'),
                                'alexnet': (['alexnet_eps0.3_layer5','alexnet_eps0.3_layer6','alexnet_eps0.3_layer7'],'alexnet_eps0.3'),
                                'hubert': (['hubert_chunk120_layer10','hubert_chunk120_layer11','hubert_chunk120_layer12'],'hubert_chunk120'),
                                'GPT2': (['GPT2_chunk120_layer10','GPT2_chunk120_layer11','GPT2_chunk120_layer12'],'GPT2_chunk120')}
if generate_subfigures:
    plotting_helpers.figure_fontsize = 6 #pt; the matrices are drawn at plotting_helpers.similarity_matrix_mm, so the four fit in 180 mm
    for axis,(features,filepath_tag) in response_similarity_features.items():
        secondlevel_ind.compute_response_similarity(load=load,response='weights',average_posterior_anterior=True,split_hemi=False,axes={axis:features},filepath_tag=filepath_tag,save_source_data=True,
                                                    plot_cbar=(axis=='GPT2')) #one colorbar, on the last matrix
        ## version with the hemispheres separate (the cached results have the hemispheres averaged):
        # secondlevel_ind.compute_response_similarity(load=load,response='weights',average_posterior_anterior=True,split_hemi=True,axes={axis:features},filepath_tag=filepath_tag)
    plotting_helpers.figure_fontsize = None

  [compute_response_similarity] CSV only has hemi='both' — overriding split_hemi=False for plotting


[run_similarity_superregion_LME] called with axis='GPT2', split_hemi=False, average_posterior_anterior=True
  results_df shape=(402, 12), columns=['subject', 'axis', 'mask1_name', 'mask2_name', 'mask1_measure_label', 'mask2_measure_label', 'hemi1', 'hemi2', 'hemi_label', 'corr', 'corr1', 'pvalue1']
  unique axes in results_df: ['GPT2']
  unique hemi1 values: ['both']
  unique hemi2 values: ['both']
  [hemi=both] axis match rows=402, hemi1 match=402, hemi2 match=402, all three=402
  [hemi=both] hemi_df after axis+hemi filter: shape=(402, 12)
  [hemi=both] mask1_super counts: {'temporal': 200, 'frontal': 100, 'MT': 68, 'STS': 34}
  [hemi=both] mask2_super counts: {'STS': 252, 'MT': 75, 'temporal': 50, 'frontal': 25}
  [hemi=both] sample mask1_name -> mask1_super: [('interact&no_interact-MT', 'MT'), ('interact&no_interact-MT', 'MT'), ('interact-no_interact-STS', 'STS'), ('interact&no_interact-MT', 'MT'), ('interact&no_interact-MT', 'MT')]
  [hemi=both] after valid_superregions filter: sha

[run_similarity_superregion_LME] called with axis='GPT2', split_hemi=False, average_posterior_anterior=True
  results_df shape=(402, 12), columns=['subject', 'axis', 'mask1_name', 'mask2_name', 'mask1_measure_label', 'mask2_measure_label', 'hemi1', 'hemi2', 'hemi_label', 'corr', 'corr1', 'pvalue1']
  unique axes in results_df: ['GPT2']
  unique hemi1 values: ['both']
  unique hemi2 values: ['both']
  [hemi=both] axis match rows=402, hemi1 match=402, hemi2 match=402, all three=402
  [hemi=both] hemi_df after axis+hemi filter: shape=(402, 12)
  [hemi=both] mask1_super counts: {'temporal': 200, 'frontal': 100, 'MT': 68, 'STS': 34}
  [hemi=both] mask2_super counts: {'STS': 252, 'MT': 75, 'temporal': 50, 'frontal': 25}
  [hemi=both] sample mask1_name -> mask1_super: [('interact&no_interact-MT', 'MT'), ('interact&no_interact-MT', 'MT'), ('interact-no_interact-STS', 'STS'), ('interact&no_interact-MT', 'MT'), ('interact&no_interact-MT', 'MT')]
  [hemi=both] after valid_superregions filter: sha

### Pooled weight PCA: interpreting visual and language predictivity in ROIs
For each localized region and model (motion, AlexNet layers 5-7, HuBERT layers 10-12, GPT-2 layers 10-12), the encoding model weights of all voxels from all participants are pooled and PCA performed on them to find the dimensions with most variance. The movie is projected onto each PC, and each PC time series is correlated with each annotated feature.

In [27]:
### pooled PCA of encoding model weights, interpreted with the annotated features
## Running from scratch is done on a cluster (needs each participant's fitted weights, analysis/EncodingModel/weights on Hugging Face), from the project root:
## for each region x model:
##   1. python -m src.pooled_pca run: z-score each voxel's weights across features, then PCA on the voxels pooled across participants
##      (the 3 layers of a model are concatenated along the feature axis, e.g. alexnet_eps0.3_layer5+alexnet_eps0.3_layer6+alexnet_eps0.3_layer7)
##   2. python -m src.pc_interpretation run --use_pooled_pca: project the movie onto the first 10 PCs and correlate each PC time series with each
##      annotated feature during speech, against a circular-shift permutation null (10,000 draws)
##      (speaking is left out: it is constant during speech, so it cannot be correlated with the PCs)
##   3. python -m src.pc_interpretation compile: compile all results and FDR correct across the 10 PCs x 14 features of each region x model
from src import pc_interpretation

results_csv = os.path.join(args_dict['out_dir'],'PCInterpretation','compiled_pooled_results.csv')
if not load:
    pc_interpretation.compile_pooled_results(os.path.join(args_dict['out_dir'],'PCInterpretation','pooled_per_pc_results'),output_file=results_csv)

## annotated features significantly correlated with any of the first 5 PCs of each localized region x model
pooled_results = pd.read_csv(results_csv)
sig = pooled_results[(pooled_results['roi_type']=='func') & (pooled_results['pc_index']<5) & (pooled_results['q_value_fdr']<0.05)]
sig = sig.assign(model=sig['feature_space'].str.split('_eps').str[0].str.split('_chunk').str[0])
print(sig.groupby(['roi','model'])['label'].unique().to_string())

roi                model           
MT                 GPT2                                                  [mentalization]
                   alexnet                                                     [valence]
                   hubert              [amplitude, music, num_agents, motion, face, s...
                   motion                                    [motion, hue, pixel, music]
STS                GPT2                            [motion, mentalization, written_text]
                   alexnet                                                     [valence]
                   hubert                 [music, amplitude, social, motion, num_agents]
                   motion              [motion, hue, pixel, amplitude, num_agents, mu...
frontal            GPT2                [motion, mentalization, num_agents, written_text]
                   SimCLR_attention                                            [valence]
                   alexnet                                                

### Generate brain maps of controlled stimuli tasks


In [ ]:
tasks = ['SIpointlights','language'] #SIpointlights contains both the motion contrast and social interaction contrast
if generate_subfigures:
    secondlevel_ind = get_secondlevel_individual(base_dir,'joint') #initialize a secondlevel individual object
    secondlevel_ind.localizer_contrasts['SIpointlights'] = ['interact&no_interact','interact-no_interact']
    secondlevel_ind.localizer_contrasts['language'] = ['intact-degraded']
    for task in tasks:
        secondlevel_ind.subjects[task] = [subject] #plot just one representative subject for each task
        secondlevel_ind.plot_localizer(task=task,symmetric_cbar=False,vmin=0,cmap='inferno_symmetric') 
        plot_secondlevel_group_glm(base_dir,task,cmap='inferno_symmetric') #plot group map for each task

### Example participant maps of each feature's product measure

In [ ]:
### Supplementary Figure 3, example participant column: each feature's product measure in the joint model
## (layers summed, vertices below 0.01 hidden after projection to the surface). The group column is made with Main Figure 2 above.
model = 'joint_chunk120'
feature_names = ['alexnet_eps0.3','motion','hubert_chunk120','GPT2_chunk120']
if generate_subfigures:
    secondlevel_ind = get_secondlevel_individual(base_dir,model) #initialize a secondlevel individual object
    secondlevel_ind.subjects['SIpointlights'] = [subject] #plot just the representative subject
    for feature in feature_names:
        secondlevel_ind.plot_map(feature=feature,measure='ind_product_measure',threshold=0.01,vmin=0,vmax=None,cmap='inferno_symmetric',png_dpi=600) #saves pdf and png

### Every participant's preference map

In [ ]:
### Supplementary Figure 4: every participant's preference map, in lateral, medial, and ventral views
## NOTE: the ROI outlines are the binary localizer masks made in the Main Figure 3A preference map cell above, so that needs to have run first
if generate_subfigures:
    secondlevel_ind = get_secondlevel_individual(base_dir,'joint_chunk120')
    features = [feature if 'layer' not in feature else feature.split('_layer')[0] for feature in secondlevel_ind.feature_names]
    file_tag = 'vent_med_lat'
    secondlevel_ind.generate_preference_maps(measure='ind_product_measure',views=['lateral','medial','ventral'],restricted=True,threshold=0.0001,features=features,file_tag=file_tag)

### Supplementary ROI analyses
Responses to the localizer conditions (Supplementary Figure 5), cross-subject encoding performance and vision vs. language predictivity (Supplementary Figure 6), voxelwise AlexNet vs. GPT-2 predictivity (Supplementary Figure 7), and feature predictivity with AlexNet and SimCLR as the image model (Supplementary Figure 8) in the individually localized regions.

In [ ]:
### supplementary ROI analyses (Supplementary Figures 5, 6, 7, and 8)
## bar plots show the mean across participants with the individual participants as points
## save_source_data saves the plotted values of each panel as a csv next to its figure
load = True ## load precomputed results while plotting
glm_localizers_to_plot = ['motion','social interaction', 'language']
glm_localizer_label_dict = {'motion':'MT','social interaction':'STS','language':'language','DMN':'STS'}
perc_top_voxels = args_dict['perc_top_voxels']

def collapse_layers(features):
    """average over all layers of any DNN (collapse all layers down to one feature name);
    the SimCLR attention and embedding layers are summarized together as SimCLR"""
    return list(dict.fromkeys(['SimCLR_eps0.3' if feature.startswith('SimCLR') else feature.split('_layer')[0] for feature in features]))

if generate_subfigures:
    plotting_helpers.figure_fontsize = 6

    ### Supplementary Figure 5: responses to the localizer conditions in each region (left and right, posterior and anterior separate) ###
    ### Supplementary Figure 7: voxelwise AlexNet vs GPT-2 product measures in each region ###
    plotting_helpers.figure_width_mm = 180
    secondlevel_ind = get_secondlevel_individual(base_dir,'joint_chunk120')
    secondlevel_ind.feature_names = collapse_layers(secondlevel_ind.feature_names)
    secondlevel_ind.plot_features = secondlevel_ind.feature_names
    secondlevel_ind.scale_by = ''
    csv_path,pkl_path = secondlevel_ind.glm_voxel_selection(load=load,include_cross_subject=False,legend_below=True,label_region=False,restrict_legend=True,average_posterior_anterior=False,stats_to_do=None,plot_stacked=False,plot_ind=True,response_label='ind_product_measure',localizers_to_plot=glm_localizers_to_plot,localizer_label_dict=glm_localizer_label_dict,filepath_tag='',figure_tag='supp',overlay_points=True,save_source_data=True)

    plotting_helpers.plot_voxelwise_feature_scatter(pkl_path=pkl_path,region_names=['MT','pSTS','aSTS','pTemp','aTemp','frontal'],
                                        feature_names=['alexnet_eps0.3','GPT2_chunk120'],
                                        feature_sources={'alexnet_eps0.3':'enc_voxelwise','GPT2_chunk120':'enc_voxelwise'},
                                        axis_labels=('Image (AlexNet) product measure ($R^2$)','Sentence (GPT-2) product measure ($R^2$)'),
                                        save_dir=os.path.join(secondlevel_ind.figure_dir,'spatial_correlation',''),file_tag='alexnet_vs_gpt2_',
                                        plot_scatter=True,split_hemisphere=False,save_source_data=True)

    ### Supplementary Figure 6A: cross-subject encoding model performance in each region ###
    ## a few participants are poorly predicted by the other participants in single regions; values beyond 3 x IQR
    ## from the quartiles of a region (Tukey's far out fence) are excluded and listed in a csv in analysis/SecondLevelIndividual
    secondlevel_ind = get_secondlevel_individual(base_dir,'joint_chunk120')
    secondlevel_ind.models = ['cross-subject encoding']
    secondlevel_ind.glm_voxel_selection(load=load,plot_lines=False,include_cross_subject=True,label_region=True,average_posterior_anterior=False,stats_to_do=None,plot_stacked=False,plot_ind=True,response_label='performance',localizers_to_plot=glm_localizers_to_plot,localizer_label_dict=glm_localizer_label_dict,filepath_tag='_for_cross_subject',overlay_points=True,exclude_outliers=True,save_source_data=True)

    ### Supplementary Figure 6B,C: vision vs language predictivity in the joint model with 1) AlexNet and 2) SimCLR as the image model ###
    for model, vision in [('joint_chunk120','vision'),('joint_chunk120_simclr','vision_simclr')]:
        secondlevel_ind = get_secondlevel_individual(base_dir,model)
        secondlevel_ind.feature_names = [vision,'language']
        secondlevel_ind.plot_features = [vision,'language']
        secondlevel_ind.glm_voxel_selection(load=load,include_cross_subject=False,label_region=False,restrict_legend=True,average_posterior_anterior=False,stats_to_do='compare_features',plot_stacked=False,plot_ind=True,response_label='ind_product_measure',localizers_to_plot=glm_localizers_to_plot,localizer_label_dict=glm_localizer_label_dict,filepath_tag='vision_vs_language',overlay_points=True,save_source_data=True)

    ### Supplementary Figure 8: feature predictivity in each region with 1) AlexNet and 2) SimCLR as the image model ###
    plotting_helpers.stacked_fig_height = 4 #wide and short, the two models are stacked on top of each other
    for model in ['joint_chunk120','joint_chunk120_simclr']:
        secondlevel_ind = get_secondlevel_individual(base_dir,model)
        secondlevel_ind.feature_names = collapse_layers(secondlevel_ind.feature_names)
        secondlevel_ind.plot_features = secondlevel_ind.feature_names
        secondlevel_ind.scale_by = ''
        secondlevel_ind.glm_voxel_selection(load=load,include_cross_subject=False,legend_below=False,label_region=False,restrict_legend=True,average_posterior_anterior=False,average_left_right=False,stats_to_do='compare_to_zero',plot_stacked=True,plot_ind=True,response_label='ind_product_measure',localizers_to_plot=glm_localizers_to_plot,localizer_label_dict=glm_localizer_label_dict,filepath_tag='',figure_tag='sep_post_ant',save_source_data=True)

    plotting_helpers.stacked_fig_height = 9
    plotting_helpers.figure_width_mm = None
    plotting_helpers.figure_fontsize = None

### Speech-only evaluation
The simplified joint encoding model (motion, AlexNet, HuBERT, word2vec, GPT-2; fit on the full movie) evaluated on only the timepoints with speaking, compared to the full movie, in each localized region.

In [ ]:
### Supplementary Figure 9: encoding model performance in the full movie vs. only the timepoints with speaking
## the simplified joint encoding model without the annotated features (so the speaking feature used to select the TRs is not in the model),
## fit on the full movie, is evaluated on only the TRs with speaking ('high' speaking condition),
## and its product measures are extracted from the same individually localized regions as in the rest of the ROI analyses
from src import feature_mask_eval
load = True ## load precomputed results while plotting
model = 'vislang_gpt2_chunk120_eps0.3'
file_label = '_encoding_model-'+model+'_smoothingfwhm-3.0_mask-ISC_chunklen-120'
roi_names = ['MT', 'pSTS', 'aSTS', 'pTemp', 'aTemp', 'frontal']
model_feature_names = ['alexnet_eps0.3', 'motion', 'hubert_chunk120', 'word2vec_chunk120', 'GPT2_chunk120']

if generate_subfigures:
    # extract the product measures of the speech-only evaluation in each region (same voxel selection as the other ROI analyses)
    feature_mask_eval.run_roi_analysis(
        base_dir=base_dir, out_dir=args_dict['out_dir'], figure_dir=args_dict['figure_dir'],
        model=model, smoothing_fwhm=3.0, mask='ISC', chunklen=120, population='NT', load=load,
        conditions=['high'], group_layers=True, roi_names=roi_names,  # the figure uses the 'high' (speaking) TRs only; the published data has only this slice
        use_glm_voxel_selection=True,
        glm_localizer_contrast={
            'MT':   ('SIpointlights', '123', 'interact&no_interact'),
            'pSTS': ('SIpointlights', '123', 'interact-no_interact'),
            'aSTS': ('SIpointlights', '123', 'interact-no_interact'),
            'pTemp': ('language','12','intact-degraded'),
            'aTemp': ('language','12','intact-degraded'),
            'frontal': ('language','12','intact-degraded')},
        merge_anterior_posterior=True)

    plotting_helpers.figure_width_mm = 120
    plotting_helpers.figure_fontsize = 6
    feature_mask_eval.plot_full_vs_subset_stacked_bars(
        subset_csv=os.path.join(args_dict['out_dir'],'SecondLevelIndividual','subset_eval','roi_summary','sub-NT'+file_label+'_roi_product_measure_glm.csv'),
        full_csv=os.path.join(args_dict['out_dir'],'SecondLevelIndividual','sub-NT_smoothingfwhm-3.0_chunklen-120_mask-ISC_model-'+model+'_perc_top_voxels-'+str(args_dict['perc_top_voxels'])+'_glm_localizer_ind_product_measure__summary.csv'),
        roi_names=roi_names,
        model_feature_names=model_feature_names,
        subset_eval_features=['speaking'],
        fig_dir=os.path.join(args_dict['figure_dir'],'subset_eval','roi_summary'),
        sid='sub-NT',
        file_label=file_label,
        features_info=feature_mask_eval._load_features_df(args_dict['out_dir'], model, 'sub-NT', file_label),
        merge_anterior_posterior=True,
        overlay_points=True,
        save_source_data=True)
    plotting_helpers.figure_width_mm = None
    plotting_helpers.figure_fontsize = None

### Dimensionality of the feature spaces 

In [ ]:
### Supplementary Figure 10: dimensionality of each feature space
## PCA of each feature space (all layers of a model concatenated), and of the combined vision (motion+AlexNet)
## and combined language (HuBERT+word2vec+GPT-2) feature spaces; plots the cumulative variance explained by the PCs
from src.intrinsic_dimensionality import IntrinsicDimensionality

if generate_subfigures:
    analyzer = IntrinsicDimensionality(
        model_list=['motion','alexnet_eps0.3','word2vec_chunk120','hubert_chunk120','GPT2_chunk120','vision','language'],
        dir=base_dir,
        concat_layers=True,
        concat_all_features=True
    )
    save_dir = os.path.join(args_dict['figure_dir'],'IntrinsicDimensionality')
    os.makedirs(save_dir, exist_ok=True)
    plotting_helpers.figure_width_mm = 90
    plotting_helpers.figure_fontsize = 6
    analyzer.run_analysis(plot_save_path=os.path.join(save_dir,'intrinsic_dim_vislang_gpt2_chunk120.pdf'))
    plotting_helpers.figure_width_mm = None
    plotting_helpers.figure_fontsize = None